# 02 — Aporte de features ya existentes

Mide cuánto suma cada grupo de features que `aplanar_anidadas` ya construye pero que `train.ipynb` todavía no usa. Modelo fijo (`HistGradientBoostingClassifier` con los mejores hiperparámetros de `train.ipynb`), misma CV, mismo `Pipeline` — la única variable entre bloques son las columnas. No entra título, TF-IDF ni features de `seller_id` (eso viene después).

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import warnings

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

from features import aplanar_anidadas, agregar_logs, CapeoPorMoneda

SEED = 42

train = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (train["condition"] == "new").astype(int)
train = aplanar_anidadas(train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

modelo_fijo = HistGradientBoostingClassifier(
    learning_rate=0.1, max_depth=5, max_iter=200, random_state=SEED
)

resultados_bloques = []


def evaluar_bloque(nombre, columnas_categoricas, columnas_numericas):
    """Arma el Pipeline (capeo -> logs -> encoding -> modelo_fijo) con las
    columnas indicadas, lo mide con la CV y guarda accuracy/roc_auc."""
    preprocesador = ColumnTransformer([
        ("onehot", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),
        ("num", "passthrough", columnas_numericas),
    ])
    pipeline = Pipeline([
        ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
        ("logs", FunctionTransformer(agregar_logs)),
        ("preprocesador", preprocesador),
        ("modelo", modelo_fijo),
    ])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        resultados = cross_validate(pipeline, train, y, cv=cv, scoring=["accuracy", "roc_auc"])

    fila = {
        "bloque": nombre,
        "accuracy_media": resultados["test_accuracy"].mean(),
        "accuracy_std": resultados["test_accuracy"].std(),
        "roc_auc_media": resultados["test_roc_auc"].mean(),
    }
    resultados_bloques.append(fila)
    print(nombre, "-> accuracy:", round(fila["accuracy_media"], 4), "+/-", round(fila["accuracy_std"], 4),
          "| roc_auc:", round(fila["roc_auc_media"], 4))
    return fila

## Bloque A — Baseline

Las 5 variables que ya usa `train.ipynb`. Debería dar ≈ 0,825 de accuracy (el número que ya mide `train.ipynb` con este mismo modelo) — si no da eso, hay que parar antes de seguir.

In [ ]:
columnas_categoricas_A = ["listing_type_id", "buying_mode"]
columnas_numericas_A = ["log_price", "log_available_quantity", "automatic_relist"]

evaluar_bloque("A - Baseline", columnas_categoricas_A, columnas_numericas_A)

## Bloque B — Cantidades

Suma `sold_quantity` e `initial_quantity` al bloque A. Ambas existen en train y test con distribuciones similares (verificado antes de empezar), no son leakage: reflejan la publicación en sí, están igual de disponibles en test.

In [ ]:
columnas_categoricas_B = columnas_categoricas_A
columnas_numericas_B = columnas_numericas_A + ["sold_quantity", "initial_quantity"]

evaluar_bloque("B - Cantidades", columnas_categoricas_B, columnas_numericas_B)

## Bloque C — Contenido de la publicación

Suma `n_fotos`, `n_atributos`, `n_variaciones` (numéricas) y `tiene_video`, `tiene_garantia` (booleanas) al bloque B.

In [ ]:
columnas_categoricas_C = columnas_categoricas_B
columnas_numericas_C = columnas_numericas_B + ["n_fotos", "n_atributos", "n_variaciones", "tiene_video", "tiene_garantia"]

evaluar_bloque("C - Contenido de la publicación", columnas_categoricas_C, columnas_numericas_C)

## Bloque D — Envío y pago

Suma `shipping_mode` (categórica nueva) y `free_shipping`, `local_pick_up`, `n_metodos_pago`, `accepts_mercadopago` (numéricas/booleanas) al bloque C.

In [ ]:
columnas_categoricas_D = columnas_categoricas_C + ["shipping_mode"]
columnas_numericas_D = columnas_numericas_C + ["free_shipping", "local_pick_up", "n_metodos_pago", "accepts_mercadopago"]

evaluar_bloque("D - Envío y pago", columnas_categoricas_D, columnas_numericas_D)

## Bloque E — Vendedor y precio

Suma `seller_state` (categórica nueva) y `es_tienda_oficial`, `tiene_descuento`, `tiene_oferta`, `tiene_parent_item` (booleanas) al bloque D.

In [ ]:
columnas_categoricas_E = columnas_categoricas_D + ["seller_state"]
columnas_numericas_E = columnas_numericas_D + ["es_tienda_oficial", "tiene_descuento", "tiene_oferta", "tiene_parent_item"]

evaluar_bloque("E - Vendedor y precio", columnas_categoricas_E, columnas_numericas_E)

## Bloque F — Tags y estado

Suma los 5 `tag_*` y `tiene_sub_status` (booleanos) al bloque E, más `status` como categórica nueva (recordar: tiene una categoría extra en train, `not_yet_active`, que no aparece en test — no rompe nada por `handle_unknown="ignore"`).

In [ ]:
tags_posibles = [
    "tag_dragged_bids_and_visits", "tag_dragged_visits", "tag_free_relist",
    "tag_good_quality_thumbnail", "tag_poor_quality_thumbnail",
]

columnas_categoricas_F = columnas_categoricas_E + ["status"]
columnas_numericas_F = columnas_numericas_E + tags_posibles + ["tiene_sub_status"]

evaluar_bloque("F - Tags y estado", columnas_categoricas_F, columnas_numericas_F)

## Tabla final y conclusión

In [ ]:
tabla_bloques = pd.DataFrame(resultados_bloques).set_index("bloque")
tabla_bloques["ganancia_accuracy"] = tabla_bloques["accuracy_media"].diff()
tabla_bloques

**Conclusión.** Comparando cada ganancia contra el desvío entre folds (que ronda 0,0015-0,0029 en todos los bloques): **B (+0,0062), C (+0,0098) y E (+0,0057) aportan de verdad** — la ganancia es 2 a 4 veces el desvío, no es casualidad del split. **D (+0,0037) aporta, pero más al límite** — la ganancia es del mismo orden que el desvío, así que hay que tomarla con cautela. **F (+0,0002) está claramente dentro del ruido** — la ganancia es un orden de magnitud menor que el desvío entre folds, o sea que tags/estado no suman nada real con este modelo.

En total, pasar de las 5 variables originales (A, accuracy 0,825) a todo lo que ya construye `aplanar_anidadas` menos tags/estado (E, accuracy 0,850) es una mejora de +0,025 en accuracy — consistente con la hipótesis inicial: el techo de 0,81-0,83 que tenían todos los modelos de `train.ipynb` lo ponían las features, no los modelos. `aplanar_anidadas` ya tenía la señal adentro; simplemente no estaba entrando al `Pipeline`.